In [16]:
import pandas as pd
import numpy as np

In [ ]:
# Cargar los archivos CSV en DataFrames
sales_df = pd.read_csv('/workspace/sales.csv')
inventories_df = pd.read_csv('/workspace/inventories.csv')
satisfaction_df = pd.read_csv('/workspace/satisfaction.csv')

# Limpiar los datos eliminando filas con valores nulos
sales_df = sales_df.dropna()
inventories_df = inventories_df.dropna()
satisfaction_df = satisfaction_df.dropna()

In [18]:
print(sales_df)
print(inventories_df)
print(satisfaction_df)

   ID_Tienda    Producto  Cantidad_Vendida  Precio_Unitario Fecha_Venta
0          1  Producto A                20              100  2023-01-05
1          1  Producto B                15              200  2023-01-06
2          2  Producto A                30              100  2023-01-07
3          2  Producto C                25              300  2023-01-08
4          3  Producto A                10              100  2023-01-09
5          3  Producto B                40              200  2023-01-10
6          4  Producto C                35              300  2023-01-11
7          4  Producto A                25              100  2023-01-12
8          5  Producto B                20              200  2023-01-13
9          5  Producto C                30              300  2023-01-14
   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización
0          1  Producto A                50          2023-01-05
1          1  Producto B                40          2023-01-06
2          2  Produ

In [19]:
# Agregar campo 'Categoria' vacío
sales_df['Categoria'] = ''

# Calcular ventas totales por producto y por tienda
ventas_totales_por_producto_tienda = sales_df.groupby(['Producto', 'ID_Tienda'])['Cantidad_Vendida'].sum().reset_index()
print("Ventas totales por producto y tienda:")
print(ventas_totales_por_producto_tienda)

# Calcular ingresos totales por tienda
sales_df['Ingresos'] = sales_df['Cantidad_Vendida'] * sales_df['Precio_Unitario']
ingresos_totales_por_tienda = sales_df.groupby('ID_Tienda')['Ingresos'].sum().reset_index()
print("\nIngresos totales por tienda:")
print(ingresos_totales_por_tienda)

# Generar resumen estadístico de las ventas
print("\nResumen estadístico de las ventas:")
print(sales_df.describe())

# Verificar si la columna 'Categoria' existe en el DataFrame
if 'Categoria' in sales_df.columns:
    promedio_ventas_categoria = (
        sales_df
        .groupby(['ID_Tienda', 'Categoria'])['Cantidad_Vendida']
        .mean()
        .reset_index()
    )
    print("\nPromedio de ventas por tienda y categoría:")
    print(promedio_ventas_categoria)

else:
    print("\nEl dataset no contiene la columna 'Categoria'.")
    
    promedio_ventas_tienda = (
        sales_df
        .groupby('ID_Tienda')['Cantidad_Vendida']
        .mean()
        .reset_index()
    )
    
    print("\nPromedio de ventas por tienda:")
    print(promedio_ventas_tienda)

# Usar groupby para ventas totales por tienda
ventas_totales_por_tienda = sales_df.groupby('ID_Tienda')['Cantidad_Vendida'].sum().reset_index()
print("\nVentas totales por tienda:")
print(ventas_totales_por_tienda)

# Ventas totales por categoría
ventas_totales_por_categoria = sales_df.groupby('Categoria')['Cantidad_Vendida'].sum().reset_index()
print("\nVentas totales por categoría:")
print(ventas_totales_por_categoria)

Ventas totales por producto y tienda:
     Producto  ID_Tienda  Cantidad_Vendida
0  Producto A          1                20
1  Producto A          2                30
2  Producto A          3                10
3  Producto A          4                25
4  Producto B          1                15
5  Producto B          3                40
6  Producto B          5                20
7  Producto C          2                25
8  Producto C          4                35
9  Producto C          5                30

Ingresos totales por tienda:
   ID_Tienda  Ingresos
0          1      5000
1          2     10500
2          3      9000
3          4     13000
4          5     13000

Resumen estadístico de las ventas:
       ID_Tienda  Cantidad_Vendida  Precio_Unitario      Ingresos
count  10.000000         10.000000        10.000000     10.000000
mean    3.000000         25.000000       190.000000   5050.000000
std     1.490712          9.128709        87.559504   3361.960407
min     1.000000     

In [20]:
# Calcular la rotación de inventarios
# Unir sales_df y inventories_df para calcular
merged_df = pd.merge(sales_df[['ID_Tienda', 'Producto', 'Cantidad_Vendida']], inventories_df[['ID_Tienda', 'Producto', 'Stock_Disponible']], on=['ID_Tienda', 'Producto'], how='inner')

# Calcular rotación: ventas totales / stock disponible
merged_df['Rotacion_Inventarios'] = merged_df['Cantidad_Vendida'] / merged_df['Stock_Disponible']

# Almacenar en inventories_df (agregar la columna)
inventories_df = pd.merge(inventories_df, merged_df[['ID_Tienda', 'Producto', 'Rotacion_Inventarios']], on=['ID_Tienda', 'Producto'], how='left')

print("DataFrame de inventarios con rotación:")
print(inventories_df)

# Filtrar tiendas con niveles críticos: porcentaje vendido < 10% (rotación < 0.1)
tiendas_criticas = merged_df[merged_df['Rotacion_Inventarios'] < 0.1]
print("\nTiendas con niveles críticos de inventario (rotación < 10%):")
print(tiendas_criticas[['ID_Tienda', 'Producto', 'Rotacion_Inventarios']])

# Usar groupby para calcular rotación promedio por tienda
rotacion_por_tienda = merged_df.groupby('ID_Tienda')['Rotacion_Inventarios'].mean().reset_index()
print("\nRotación promedio de inventarios por tienda:")
print(rotacion_por_tienda)

DataFrame de inventarios con rotación:
   ID_Tienda    Producto  Stock_Disponible Fecha_Actualización  \
0          1  Producto A                50          2023-01-05   
1          1  Producto B                40          2023-01-06   
2          2  Producto A                60          2023-01-07   
3          2  Producto C                45          2023-01-08   
4          3  Producto A                30          2023-01-09   
5          3  Producto B                80          2023-01-10   
6          4  Producto C                70          2023-01-11   
7          4  Producto A                50          2023-01-12   
8          5  Producto B                40          2023-01-13   
9          5  Producto C                60          2023-01-14   

   Rotacion_Inventarios  
0              0.400000  
1              0.375000  
2              0.500000  
3              0.555556  
4              0.333333  
5              0.500000  
6              0.500000  
7              0.500000  


In [21]:
# Análisis de la satisfacción de los clientes en cada tienda
# Relacionar con el rendimiento de las ventas
analisis_satisfaccion = pd.merge(satisfaction_df, ventas_totales_por_tienda, on='ID_Tienda', how='left')
analisis_satisfaccion = pd.merge(analisis_satisfaccion, ingresos_totales_por_tienda, on='ID_Tienda', how='left')

print("Análisis de satisfacción y rendimiento de ventas:")
print(analisis_satisfaccion)

# Filtrar tiendas con niveles bajos de satisfacción (< 60%)
tiendas_bajas_satisfaccion = analisis_satisfaccion[analisis_satisfaccion['Satisfacción_Promedio'] < 60]
print("\nTiendas con niveles bajos de satisfacción (< 60%):")
print(tiendas_bajas_satisfaccion)

# Recomendaciones para mejorar el rendimiento
print("\nRecomendaciones para tiendas con baja satisfacción:")
for index, row in tiendas_bajas_satisfaccion.iterrows():
    tienda_id = row['ID_Tienda']
    satisfaccion = row['Satisfacción_Promedio']
    ventas = row['Cantidad_Vendida']
    ingresos = row['Ingresos']
    print(f"Tienda {tienda_id} (Satisfacción: {satisfaccion}%):")
    print("  - Mejorar la atención al cliente y el servicio.")
    print("  - Revisar y optimizar los niveles de inventario.")
    print("  - Ofrecer promociones o descuentos para aumentar las ventas.")
    print("  - Entrenar al personal en técnicas de venta.")
    print()

Análisis de satisfacción y rendimiento de ventas:
   ID_Tienda  Satisfacción_Promedio Fecha_Evaluación  Cantidad_Vendida  \
0          1                     85       2023-01-15                35   
1          2                     90       2023-01-15                55   
2          3                     70       2023-01-15                50   
3          4                     65       2023-01-15                60   
4          5                     55       2023-01-15                50   

   Ingresos  
0      5000  
1     10500  
2      9000  
3     13000  
4     13000  

Tiendas con niveles bajos de satisfacción (< 60%):
   ID_Tienda  Satisfacción_Promedio Fecha_Evaluación  Cantidad_Vendida  \
4          5                     55       2023-01-15                50   

   Ingresos  
4     13000  

Recomendaciones para tiendas con baja satisfacción:
Tienda 5 (Satisfacción: 55%):
  - Mejorar la atención al cliente y el servicio.
  - Revisar y optimizar los niveles de inventario.
  - Ofre

In [22]:
# Usar Numpy para cálculos sobre las ventas
# Asumiendo 'Cantidad_Vendida' como ventas totales por tienda
ventas_array = ventas_totales_por_tienda['Cantidad_Vendida'].to_numpy()

# Mediana de las ventas totales
mediana_ventas = np.median(ventas_array)
print(f"Mediana de las ventas totales: {mediana_ventas}")

# Desviación estándar de las ventas totales
desviacion_ventas = np.std(ventas_array)
print(f"Desviación estándar de las ventas totales: {desviacion_ventas}")

# Generar arrays aleatorios para simular proyecciones de ventas futuras
np.random.seed(42)  # Establecer semilla para reproducibilidad
proyecciones_ventas = np.random.normal(loc=mediana_ventas, scale=desviacion_ventas, size=(5, 10))  # 5 escenarios, 10 periodos
print("\nProyecciones de ventas futuras (arrays aleatorios):")
print(proyecciones_ventas)

# Calcular estadísticas básicas sobre las proyecciones
media_proyecciones = np.mean(proyecciones_ventas)
mediana_proyecciones = np.median(proyecciones_ventas)
desviacion_proyecciones = np.std(proyecciones_ventas)
print(f"\nEstadísticas de las proyecciones:")
print(f"Media: {media_proyecciones}")
print(f"Mediana: {mediana_proyecciones}")
print(f"Desviación estándar: {desviacion_proyecciones}")

Mediana de las ventas totales: 50.0
Desviación estándar de las ventas totales: 8.366600265340756

Proyecciones de ventas futuras (arrays aleatorios):
[[54.15580876 48.84319786 55.41895109 62.742582   48.04093231 48.04106967
  63.21264236 56.42081961 46.07209548 54.539383  ]
 [46.12276941 46.10342532 52.02440161 33.992349   35.568302   45.29556501
  41.52604688 52.62918182 42.40292553 38.18381948]
 [62.26249738 48.11101994 50.5649815  38.07970145 45.44536735 50.92804497
  40.37009683 53.14331514 44.97469618 47.559515  ]
 [44.9657613  65.49727115 49.88707412 41.15055546 56.88190448 39.78568919
  51.74747821 33.60422342 38.88759825 51.64705927]
 [56.17845468 51.43376991 49.03241705 47.48078574 37.62979752 43.97735126
  46.14601954 58.8445191  52.87491687 35.24934777]]

Estadísticas de las proyecciones:
Media: 48.11354996445656
Mediana: 48.041000993386106
Desviación estándar: 7.7331226064538905
